# 세션 2 — PDF에서 글자 꺼내기(추출)와 Base RAG

RAG의 첫 단계는 PDF를 텍스트로 만드는 것(추출)입니다. 같은 PDF라도 방법에 따라 표와 숫자가
살아남는 정도가 다르고, 그것이 답변 정확도로 이어집니다. 네 가지 추출 방법을 비교하고 그 결과로 첫 RAG를 만듭니다.

| 방법 | 설명 | 표 보존 | 비용 |
|------|------|---------|------|
| PyMuPDF | 텍스트 레이어 그대로 | 약함 | 무료·빠름 |
| PyMuPDF4LLM | 레이아웃을 Markdown으로 | 보통 | 무료·빠름 |
| docling | AI 모델로 표 구조 인식 | 좋음 | 무료·로컬 |
| VLM (GPT-4o) | 페이지를 이미지로 읽음 | 매우 좋음 | 유료·느림 |

`REBUILD=False` 면 동봉된 캐시를 쓰고, `True` 면 본인 API 키로 추출~인덱스를 직접 만듭니다(VLM 40쪽, 몇 분).

In [1]:
# 준비 셀 — 경로와 API 키 (매 세션 같은 코드로 시작한다)
import os, time, pickle
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()                            # .env 의 OPENAI_API_KEY 를 환경변수로 등록

ROOT = Path.cwd().parent                 # notebooks/ 의 상위 = 프로젝트 폴더
PDF = ROOT / "data" / "저출생_반전을_위한_대책_관계부처_합동_.pdf"
CACHE = ROOT / "cache"
CACHE.mkdir(exist_ok=True)

print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")), "| PDF:", PDF.exists())

OpenAI key: True | PDF: True


In [17]:
# 임베딩 모델과 LLM 준비
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

# bge-m3: 문장 → 1024차원 벡터. 비슷한 뜻이면 가까운 벡터가 된다 (처음엔 약 2GB 다운로드)
# HuggingFace에는 오픈소스 모델이 많다고 함.
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},                  # GPU 없이 CPU로 동작
    encode_kwargs={"normalize_embeddings": True},    # 벡터 길이를 1로 통일 → 비교가 공정해진다
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # temperature=0: 재현성, 높아질수록 창의적이라고 함
# 실무에서 temperatur를 건드리는 경우는 거의 없다고 함

In [ ]:
# RAG의 뼈대 — !!!!검색된 청크!!!!를 {context} 자리에 끼워 LLM에게 보낸다 
RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context} 

질문: {question}
답변:"""

def ask(question, retriever):
    # RAG 3단계: ① 검색 → ② 프롬프트에 끼우기 → ③ LLM 생성
    docs = retriever.invoke(question)                          # ① 질문과 비슷한 청크 찾기
    context = "\n---\n".join(d.page_content for d in docs)     # ② 청크들을 구분선으로 이어 붙이기
    answer = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content   # ③ 생성
    return answer, docs                                        # 답변과 근거 청크를 함께 반환

# show_docs : 어디서 나온 내용인지 확인할 수 있도록 청크를 보기 좋게 출력하는 함수

def show_docs(docs, title="검색 결과"):
    # 검색된 청크를 "몇 쪽에서 나온 어떤 내용인지" 보기 좋게 출력한다
    print(title)
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")         # 청크에 저장된 쪽 번호 (없으면 ?)
        print(f"  {i}. (p.{page}) {d.page_content[:100].strip()} ...")

In [4]:
# True 로 바꾸면 캐시 대신 본인 API 키로 직접 추출·임베딩한다 (시간·소액 과금)
REBUILD = False

## 2-1. 같은 페이지를 네 방법으로 읽어보기

'현행 → 개선' 대비표가 실제로 있는 페이지(32쪽, 신혼·출산가구 주택공급 표)를 네 방법으로 뽑아
표가 얼마나 살아남는지 봅니다. docling과 VLM은 모델·네트워크를 쓰므로 비교는 한 페이지만 합니다.

In [20]:
# 방법 1: PyMuPDF — PDF의 텍스트 레이어를 그대로 꺼낸다 (가장 단순·빠름)
PAGE = 31          # 0부터 세므로 32쪽 — 구분|현행|개선 대비표가 있는 페이지

from langchain_community.document_loaders import PyMuPDFLoader
pdf_pages = PyMuPDFLoader(str(PDF)).load()     # 쪽마다 Document 하나씩, 총 40개
print(pdf_pages[PAGE].page_content[:200])
print("\n표 구분자 | 포함:", "|" in pdf_pages[PAGE].page_content)   # 표가 살아남았는지 확인

- 56 -
구분
현행
개선
자격
요건
특별공급
공공 민간
‣생애기간 중  특별공급 1회 당첨 제한
‣신규 출산가구는 특별공급 재당첨 1회 허용
자격
요건
특별공급
공공 민간
‣(청약 이력) 배우자의 결혼 前 청약
당첨 이력 배제
‣본인 및 배우자의 결혼 前 청약당첨 
이력 배제(본인의 경우 신혼 특공만 해당)
자격
요건
특별공급
공공 민간
‣(무주택 조건

표 구분자 | 포함: False


In [21]:
# 방법 2: PyMuPDF4LLM — 레이아웃을 분석해 Markdown(표는 | 구분)으로 변환한다
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
md_pages = PyMuPDF4LLMLoader(str(PDF)).load()
print(md_pages[PAGE].page_content[:600])
print("\n표 구분자 | 포함:", "|" in md_pages[PAGE].page_content)

|구분|Col2|현행|개선|
|---|---|---|---|
|**자격**<br>**요건**|**특별공급**<br>**공공 민간**|‣생애기간 중  특별공급 1회 당첨 제한|‣**신규 출산가구는 특별공급 재당첨 1회 허용**|
|**자격**<br>**요건**|**특별공급**<br>**공공 민간**|‣(청약 이력) 배우자의 결혼 前 청약<br>당첨 이력 배제|‣**본인 및 배우자**의 결혼 前**청약당첨**<br>**이력 배제**(본인의 경우 신혼 특공만 해당)|
|**자격**<br>**요건**|**특별공급**<br>**공공 민간**|‣(무주택 조건) 혼인신고~ 입주자<br>모집공고 시까지 충족 필요|<br>‣**입주자모집공고 시**에만 충족 여부 확인|
|**소득**<br>**요건**|**일반공급**<br>**공공**|‣도시근로자 월평균 100%|‣순차제**140%**, 추첨제** 200%**로<br>**맞벌이 소득기준** 신설|



**‧**
➌ (공공임대 거주지원 강화) 출산가구 대상 공공임대 재계약 소득 자산

기준폐지 및 평형 상향 지원, 장기전세주택 제도개선 (국토부)


‘
 24년 이후 신규 출산가구 (임신 포함) 에 대해 자녀가 성년이 될


표 구분자 | 포함: True


In [7]:
# 방법 3: docling — AI 모델이 표 구조를 인식해 복원한다 (무료·로컬, 대신 느림)
import logging
from docling.document_converter import DocumentConverter

logging.disable(logging.INFO)   # OCR 진행 로그 숨김 (경고·오류는 표시됨)

converter = DocumentConverter()
# docling의 page_range는 1부터 세므로 PAGE+1
docling_md = converter.convert(str(PDF), page_range=(PAGE + 1, PAGE + 1)).document.export_to_markdown()
print(docling_md[:800])
print("\n표 구분자 | 포함:", "|" in docling_md)

| 구분      | 구분                                        | 현행                                                 | 개선                                                                        |
|-----------|---------------------------------------------|------------------------------------------------------|-----------------------------------------------------------------------------|
| 자격 요건 | 특별공급 공공 민간 ‣ 생애기간 중            | 특별공급 1회 당첨 제한                               | ‣ 신규출산가구는특별공급 재당첨 1회 허용                                    |
| 자격 요건 | 특별공급 공공 민간 ‣                        | (청약 이력) 배우자의 결혼 前 청약 당첨 이력 배제 ‣   | 본인 및 배우자 의 결혼 前 청약당첨 이력 배제 (본인의 경우 신혼 특공만 해당) |
| 자격 요건 | 특별공급 공공 민간 ‣ (무주택 조건) 모집공고 | 혼인신고~입주자 시까지 충족 필요 ‣ 입주자모집공고 시 | 에만 충족 여부 확인                                                         

표 구분자 | 포함: True


In [ ]:
# 방법 4: VLM — 페이지를 "이미지"로 만들어 GPT-4o에게 읽어 달라고 한다 (유료·가장 느림)
import base64, fitz
from langchain_core.messages import HumanMessage

def vlm_read_page(pdf_path, page_index):
    # ① 페이지를 PNG 이미지로 렌더링 → ② base64 문자열로 인코딩 → ③ GPT-4o에 전송
    doc = fitz.open(pdf_path)
    image_png = doc[page_index].get_pixmap(dpi=400).tobytes("png")   # dpi(dot per inch)가 높을수록 선명·용량 증가 
    doc.close()
    b64 = base64.b64encode(image_png).decode()      # 이미지를 API로 보내기 위한 텍스트 변환
    vlm = ChatOpenAI(model="gpt-4o", temperature=0) # 이미지를 읽으려면 gpt-4o 같은 비전 모델 필요
    msg = HumanMessage(content=[                    # 텍스트 지시 + 이미지를 한 메시지에 담는다
        {"type": "text", "text": "이 페이지의 글과 표를 Markdown으로 정확히 옮겨줘. 표는 | 로."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
    ])
    return vlm.invoke([msg]).content

# gpt-4o 호출 1회 — 키에 gpt-4o 권한이 없으면 이 데모만 건너뛴다 (아래 셀들은 캐시로 동작)
try:
    vlm_page = vlm_read_page(str(PDF), PAGE)
    print(vlm_page[:500])
    print("\n표 구분자 | 포함:", "|" in vlm_page)
except Exception as e:
    print(f"VLM 호출 실패({type(e).__name__}) — 키의 gpt-4o 권한을 확인하세요.")
    print("이 데모는 건너뛰어도 됩니다. 저장된 출력을 참고하세요.")

```markdown
| 구분 | 현행 | 개선 |
| --- | --- | --- |
| 자연 출산율 | 생애기간 중 특공 1회 당첨 제한 | 신규 출산가구는 특별공급 재당첨 1회 허용 |
| 입주권 | (청약이력) 배우자가 결혼 후 청약 당첨 이력 없게 | 본인 및 배우자가 결혼 후 청약당첨 이력 없게(결혼이전 경우 새로 포함) |
| 자격 | (입주자모집공고) 신혼 특공 시 입주자모집공고 시까지 혼인신고 완료 | 입주자모집공고 시이혼 중복 여부 확인 |
| 소득 | 도시근로자 가구당 월평균 소득 100% | *추자시 140%, 공공시 200%로 맞벌이 소득기준 신설 |

- (공공임대 거주지원 강화) 출산가구 대상 공공임대 재계약 소득·자산 기준폐지 및 평형 상향 지원, 장기전세주택 제도개선(국토부)
  - '24년 이후 신규 출산가구(입신 포함)에 대해 자녀가 성년이 될 때까지 소득·자산 무관하게 공공임대 재계약 허용(최대 20년)
  - 2세 이하 자녀 가구가 희망할 경우 

표 구분자 | 포함: True


단순 텍스트(PyMuPDF)는 '구분|현행|개선' 표가 줄글로 뭉개지고(`|` 포함: False),
PyMuPDF4LLM·docling·VLM은 표를 `|` 구분자로 복원합니다(`|` 포함: True).
표가 이미지로만 있는 문서일수록 이 차이가 정확도를 좌우합니다.

다만 이 PDF는 디지털로 만든 파일이라 텍스트가 깨끗해서, 사실 단순 추출로도 꽤 정확합니다.
docling·VLM의 위력은 스캔본이나 복잡한 표 문서에서 드러납니다. (2-4에서 점수로 확인합니다)

## 2-2. 청크로 자르고 검색 인덱스(FAISS) 만들기 (== Vector DB 만들기)

표 보존이 가장 좋은 VLM 추출본을 기본 코퍼스로 씁니다. 전체 40쪽 VLM 추출은 시간이 걸리므로
캐시를 쓰거나 `REBUILD=True` 로 직접 굽습니다.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 긴 문서를 검색 단위(청크)로 자른다.
#   chunk_size=700  : 청크 하나가 최대 700자 — 너무 크면 노이즈, 너무 작으면 맥락 상실
#   chunk_overlap=180: 이웃 청크와 180자를 겹치게 — 경계에서 문장이 끊기는 것을 막는 보험
splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=180)

In [25]:
# VLM 추출본으로 기본 코퍼스(청크 + FAISS 인덱스)를 준비한다
from langchain_community.vectorstores import FAISS

try:
    if REBUILD:
        # 느린 길: 40쪽 전부를 VLM으로 다시 읽고 인덱스도 새로 만든다
        print("VLM으로 40쪽 추출 중... (gpt-4o 40콜, 몇 분)")
        doc = fitz.open(PDF)
        from langchain_core.documents import Document
        vlm_docs = [Document(page_content=vlm_read_page(str(PDF), i), metadata={"page": i + 1})
                    for i in range(doc.page_count)]          # 쪽마다 VLM 호출 1회
        doc.close()
        splits = splitter.split_documents(vlm_docs)          # 청크로 자르고
        pickle.dump(splits, open(CACHE / "splits.pkl", "wb"))    # 다음을 위해 저장
        db = FAISS.from_documents(splits, embeddings)        # 청크 전체를 임베딩해 인덱스 생성
        db.save_local(str(CACHE / "faiss_index"))
    else:
        # 빠른 길: 동봉된 캐시를 로드만 한다 (수 초)
        splits = pickle.load(open(CACHE / "splits.pkl", "rb"))
        # FAISS 인덱스는 pickle 기반 → 직접 만든(신뢰할 수 있는) 파일만 이 옵션으로 연다
        db = FAISS.load_local(str(CACHE / "faiss_index"), embeddings,
                              allow_dangerous_deserialization=True)
except Exception as e:
    # 폴백: VLM 실패(gpt-4o 권한 없음)나 캐시 없음 → PyMuPDF4LLM 추출본으로 대체
    print(f"VLM 코퍼스 준비 실패({type(e).__name__}) → PyMuPDF4LLM 추출본으로 대체합니다.")
    splits = splitter.split_documents(md_pages)
    db = FAISS.from_documents(splits, embeddings)

# retriever: "질문을 주면 비슷한 청크 k개를 돌려주는" 검색기. k=5 → 상위 5개 사용
retriever = db.as_retriever(search_kwargs={"k": 5})
print("청크", len(splits), "개, 인덱스 벡터", db.index.ntotal, "개")

청크 181 개, 인덱스 벡터 181 개


## 2-3. Base RAG 한 번 돌려보기
VLM이 만든 청크를 가지고 Vector DB를 만들기?? \n
질문 → 비슷한 청크 5개 검색 → 그 내용으로 LLM이 답변하는 흐름입니다.

In [11]:
# 첫 RAG 실행 — 질문 하나를 던져 검색 근거와 답변을 함께 본다
question = "육아휴직 급여 최대 상한은 얼마로 인상되나?"
answer, docs = ask(question, retriever)      # ask 는 위(공통 헬퍼)에서 만든 함수
show_docs(docs[:3])                          # 어떤 청크를 근거로 썼는지 상위 3개만 표시
print("\n질문:", question)
print("답변:", answer)

검색 결과
  1. (p.18) ```markdown
### ② (휴직 유연화) 필요한 때 유연하게 활용할 수 있도록 육아휴직 분할사용 횟수 확대(고용부)

- 단기 육아휴직 사용은 육아휴직 분할횟수 산정 시 차 ...
  2. (p.2) 2. **소득 걱정 없이 누구나 사용할 수 있도록 개선**
   - 육아휴직 급여 인상(하한 150만 원 → 최대 250만 원)
   - 우선육아 수당 추가 등 시기에 상대적으로 ...
  3. (p.18) ### ⑤ (사후지급금 폐지) 육아휴직급여의 25%를 복직 후 6개월 이상 계속 근로 시 지급하는 사후지급금을 폐지하여 소득대체율을 인상(고용부)

---

## ② 임신·육아기 ...

질문: 육아휴직 급여 최대 상한은 얼마로 인상되나?
답변: 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.


## 2-4. 추출 방법에 따라 점수가 달라질까

네 추출본으로 각각 평가셋 12문항을 풀어, 어떤 질문을 맞고 틀렸는지 나란히 봅니다.
4개 추출본과 파서별 FAISS 인덱스는 `REBUILD=False` 면 캐시에서 로드하고(수 초),
`True` 면 새로 만들어 저장합니다 — 세션 3·4가 이 캐시를 그대로 재사용합니다.

> 채점 셀은 4개 추출본 × 12문항 = **gpt-4o-mini 48콜**이라 약 2~3분 걸립니다(API 과금 발생).
> 강의에서는 셀을 실행해 두고 그동안 설명을 진행합니다.

In [12]:
# 정답이 문서에서 확인된 평가 문항 12개 (세션 1에서 만든 것)
EVAL = [
    {"q": "2023년 한국 합계출산율은?",                         "a": "0.72명"},
    {"q": "2023년 출생아 수는?",                               "a": "23만명"},
    {"q": "정부가 2030년까지 목표로 하는 합계출산율은?",        "a": "1.0"},
    {"q": "육아휴직 급여 최대 상한은 얼마로 인상되나?",         "a": "250만원"},
    {"q": "아빠(배우자) 출산휴가 기간은 며칠로 확대되나?",      "a": "20일"},
    {"q": "부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?", "a": "1년 6개월"},
    {"q": "대체인력지원금은 월 얼마로 인상되나?",              "a": "120만원"},
    {"q": "임기 내 공공보육 이용률 목표는?",                   "a": "50%"},
    {"q": "0세반 교사 대 영아 비율은 어떻게 개선되나?",        "a": "1:2"},
    {"q": "'25년 이후 출산가구 신생아 특례대출 소득요건은?",    "a": "2.5억원"},
    {"q": "출산가구 대상 주택공급은 연 몇 호로 확대되나?",      "a": "12만호"},
    {"q": "보고서가 제시한 한국의 육아휴직 소득대체율은?",      "a": "38.6%"},
]

def is_correct(answer, expected):
    # 표기 차이를 흡수한다 ("월 250만원" == "250만원", "20근무일" == "20일") — 세션 1에서 설명
    def norm(s):
        s = s.replace(" ", "").replace(",", "").replace("근무일", "일")
        return s[1:] if s.startswith("월") else s
    return norm(expected) in norm(answer)

In [13]:
# 4개 추출본 각각의 청크와 FAISS 인덱스를 준비한다
from langchain_core.documents import Document

if REBUILD:
    # 느린 길: docling 전체 변환 + 추출본 4개를 전부 임베딩 (수 분)
    print("docling 전체 추출 + 인덱스 4개 생성 중... (수 분)")
    docling_full = converter.convert(str(PDF)).document.export_to_markdown()
    parser_splits = {                            # 추출본 이름 → 청크 리스트
        "pymupdf": splitter.split_documents(pdf_pages),
        "4llm":    splitter.split_documents(md_pages),
        "docling": splitter.split_documents([Document(page_content=docling_full, metadata={"source": "docling"})]),
        "vlm":     splits,                       # 2-2에서 만든 VLM 청크 재사용
    }
    pickle.dump(parser_splits, open(CACHE / "splits_all.pkl", "wb"))
    retrievers = {}
    for n, sp in parser_splits.items():
        db_n = FAISS.from_documents(sp, embeddings)      # 추출본마다 인덱스 하나씩
        db_n.save_local(str(CACHE / f"idx_{n}"))
        retrievers[n] = db_n.as_retriever(search_kwargs={"k": 5})
else:
    # 빠른 길: 사전 계산된 4개 추출본과 인덱스를 로드만 한다 (수 초)
    parser_splits = pickle.load(open(CACHE / "splits_all.pkl", "rb"))
    retrievers = {
        n: FAISS.load_local(str(CACHE / f"idx_{n}"), embeddings,
                            allow_dangerous_deserialization=True).as_retriever(search_kwargs={"k": 5})
        for n in parser_splits
    }

print("청크 수:", {n: len(sp) for n, sp in parser_splits.items()})

청크 수: {'pymupdf': 112, '4llm': 122, 'docling': 140, 'vlm': 181}


In [14]:
# 4개 추출본 × 12문항 = 48콜 채점 — 결과를 O/X 표로 정리한다 (약 2~3분)
names = list(retrievers)
results = []                                 # 문항별 상세(질문·정답·추출본별 답변)를 모아 둔다
totals = {n: 0 for n in names}               # 추출본별 맞힌 개수
for item in EVAL:
    answers = {}
    for n in names:                          # 같은 질문을 4개 추출본에 각각 던진다
        answer, _ = ask(item["q"], retrievers[n])
        ok = is_correct(answer, item["a"])
        totals[n] += ok
        answers[n] = (answer, ok)
    results.append({"q": item["q"], "a": item["a"], "answers": answers})

# 표 출력 — 행: 문항 정답, 열: 추출본, 칸: O/X
print(f"{'정답':<9}", "".join(f"{n:>9}" for n in names))
for r in results:
    print(f"{r['a']:<9}", "".join(f"{'O' if r['answers'][n][1] else 'X':>9}" for n in names))
print(f"{'합계':<9}", "".join(f"{str(totals[n])+'/12':>9}" for n in names))

정답          pymupdf     4llm  docling      vlm
0.72명             O        O        O        O
23만명              O        O        O        O
1.0               O        O        X        O
250만원             O        O        O        O
20일               O        O        O        O
1년 6개월            O        O        O        O
120만원             O        O        O        O
50%               O        O        O        X
1:2               O        O        O        O
2.5억원             O        O        O        X
12만호              O        O        O        O
38.6%             O        X        O        X
합계            12/12    11/12    11/12     9/12


### 실제 답변과 정답 비교

O/X만으로는 왜 틀렸는지 알 수 없습니다. 각 파서가 실제로 뭐라고 답했는지 정답과 나란히 봅니다.

In [15]:
# O/X 뒤에 숨은 실제 답변을 문항별로 나란히 본다 (추가 API 호출 없음 — 저장해 둔 results 사용)
for r in results:
    print(f"Q. {r['q']}  (정답: {r['a']})")
    for n in names:
        answer, ok = r["answers"][n]
        print(f"   {n:>8} {'O' if ok else 'X'} {answer[:80]}")
    print()

Q. 2023년 한국 합계출산율은?  (정답: 0.72명)
    pymupdf O 2023년 한국의 합계출산율은 0.72명입니다.
       4llm O 2023년 한국의 합계출산율은 0.72명입니다.
    docling O 2023년 한국의 합계출산율은 0.72명입니다.
        vlm O 2023년 한국의 합계출산율은 0.72명입니다.

Q. 2023년 출생아 수는?  (정답: 23만명)
    pymupdf O 2023년 출생아 수는 23만명입니다.
       4llm O 2023년 출생아 수는 23만명입니다.
    docling O 2023년 출생아 수는 23만명입니다.
        vlm O 2023년 출생아 수는 23만명입니다.

Q. 정부가 2030년까지 목표로 하는 합계출산율은?  (정답: 1.0)
    pymupdf O 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
       4llm O 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
    docling X 정부가 2030년까지 목표로 하는 합계출산율에 대한 구체적인 수치는 맥락에 명시되어 있지 않습니다. 따라서 해당 정보는 제공할 수 없습니다.
        vlm O 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.

Q. 육아휴직 급여 최대 상한은 얼마로 인상되나?  (정답: 250만원)
    pymupdf O 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
       4llm O 육아휴직 급여 최대 상한은 250만원으로 인상됩니다.
    docling O 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
        vlm O 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.

Q. 아빠(배우자) 출산휴가 기간은 며칠로 확대되나?  (정답: 20일)
    pymupdf O 아빠(배우자) 출산휴가 기간은 10일에서 20근무일로 확대됩니다.
       4llm O 아

## 2-5. 왜 더 좋은 파서가 더 낮게 나올까

표 보존이 가장 좋은 VLM이 오히려 점수가 낮게 나오기도 합니다. 이유는 분명합니다.

"더 좋다"는 것은 깨진 표를 복원하는 능력입니다. 이 PDF는 텍스트가 깨끗해서 PyMuPDF가 그대로 복사해도
이미 정확합니다. 복원할 게 없는데 VLM은 페이지를 다시 읽으며 오히려 손해를 봅니다.

- 재해석 오독: `2→2.5억` 을 VLM이 `2~25억` 으로 잘못 읽기도 합니다. PyMuPDF는 그대로 복사하니 맞습니다.
- 재구조화: VLM 청크가 더 잘게 쪼개지면 숫자가 라벨과 분리되거나, 정답 청크가 상위 5개에 들지 못합니다.

정리하면 무조건 좋은 파서는 없고, 문서 성격에 맞춰 고릅니다. 깨끗한 디지털 PDF는 단순 추출이 유리하고,
스캔본·복잡표는 docling·VLM이 유리합니다. 지금은 FAISS 단독이라 출렁임이 큰데, 다음 세션의 하이브리드 검색으로 안정화됩니다.

## 직접 해보기

> 예시 코드를 **새 셀에 복사해** 실행해 보세요.

**1. `question` 을 다른 평가셋 문항으로 바꿔 답을 확인해 보세요.**
```python
question = "임기 내 공공보육 이용률 목표는?"
answer, docs = ask(question, retriever)
show_docs(docs[:3])
print("답변:", answer)
```

**2. `k` 를 2나 8로 바꾸면 답이 어떻게 달라질까요?**
```python
retriever_k2 = db.as_retriever(search_kwargs={"k": 2})      # 근거 2개만 사용
print("k=2:", ask("보고서가 제시한 한국의 육아휴직 소득대체율은?", retriever_k2)[0][:80])

retriever_k8 = db.as_retriever(search_kwargs={"k": 8})      # 근거 8개 사용
print("k=8:", ask("보고서가 제시한 한국의 육아휴직 소득대체율은?", retriever_k8)[0][:80])
```

**3. `REBUILD=True` 로 직접 VLM 추출을 돌려 보세요.** 맨 위 `REBUILD = False` 셀을 `True` 로 바꾼 뒤
상단부터 다시 실행하면 됩니다. (gpt-4o 40콜 — 몇 분 + 소액 과금이 발생하니 쉬는 시간에 권장)

## 시간이 남으면 (심화) — 청크 크기가 점수를 움직일까

지금까지는 700자/겹침 180자로 잘랐습니다. 크기를 바꾸면 어떻게 될까요?

- **작게(300자)** — 청크가 정밀해지지만, 표나 문단이 중간에 끊겨 맥락을 잃기 쉽습니다.
- **크게(1200자)** — 맥락은 살지만, 한 청크에 여러 주제가 섞여 검색이 둔해지고 노이즈가 늘어납니다.

PyMuPDF4LLM 추출본으로 두 설정을 다시 채점해 기준(700자 → 11/12)과 비교합니다.
참고로 pymupdf 추출본은 이 문서에서 워낙 견고해 어느 크기든 12/12가 나옵니다 —
**추출이 깨끗할수록 청킹의 여유도 커집니다.**

> 임베딩 재계산 + 24콜 — 약 2~4분.

In [16]:
# 청크 크기 두 가지(300자·1200자)로 PyMuPDF4LLM 추출본을 다시 채점한다
for size, overlap in [(300, 80), (1200, 300)]:
    # 새 크기로 다시 자르고 → 다시 임베딩(인덱스 생성) → 12문항 채점
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap).split_documents(md_pages)
    r_t = FAISS.from_documents(sp, embeddings).as_retriever(search_kwargs={"k": 5})
    misses = []                                  # 틀린 문항의 정답을 모아 둔다
    for item in EVAL:
        if not is_correct(ask(item["q"], r_t)[0], item["a"]):
            misses.append(item["a"])
    print(f"chunk_size={size:>5} / overlap={overlap:>4} → 청크 {len(sp):>4}개, "
          f"점수 {12-len(misses)}/12  틀린 문항: {misses or '없음'}")

print(f"chunk_size=  700 / overlap= 180 → 청크 {len(parser_splits['4llm']):>4}개, 점수 11/12  (2-4의 4llm 열)")
# 다른 크기(500? 2000?)도 넣어 보세요. 이 문서·추출본의 최적점은 어디일까?

chunk_size=  300 / overlap=  80 → 청크  270개, 점수 11/12  틀린 문항: ['38.6%']
chunk_size= 1200 / overlap= 300 → 청크   74개, 점수 10/12  틀린 문항: ['120만원', '38.6%']
chunk_size=  700 / overlap= 180 → 청크  122개, 점수 11/12  (2-4의 4llm 열)


## 정리
- 추출이 RAG의 상한을 정합니다. 표가 깨지면 숫자를 맞힐 수 없습니다.
- 다만 문서 성격이 중요합니다. 깨끗한 디지털 PDF는 단순 추출로도 충분합니다.
- 청크 크기도 점수를 움직입니다(심화) — 추출이 깨끗할수록 청킹의 여유가 커집니다.
- `cache/splits_all.pkl`, `idx_*` 는 세션 3·4에서 그대로 씁니다.